# 06 — Detecção de Anomalias em Aeroportos

Objetivo: identificar aeroportos com comportamento **atípico** em relação ao perfil de atraso — aqueles que não se encaixam bem em nenhum cluster e apresentam padrões fora do esperado.

Abordagens utilizadas:
1. **Isolation Forest** — detecta outliers por isolamento aleatório de pontos
2. **Local Outlier Factor (LOF)** — detecta anomalias por densidade local
3. **Silhouette individual** — aeroportos mal alocados pelo KMeans (notebook 04)
4. **Consenso** — aeroportos sinalizados por múltiplos métodos ao mesmo tempo

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples

RANDOM_STATE = 42
plt.rcParams["figure.dpi"] = 110

## 1. Agregação por aeroporto

Mesmas features do notebook 04 para manter consistência.

In [ ]:
df = pl.read_parquet("../data/processed/flights_model.parquet")

MIN_VOOS = 500

airport_features = (
    df
    .filter(pl.col("ARRIVAL_DELAY").is_not_null())
    .group_by("ORIGIN_AIRPORT")
    .agg([
        pl.len().alias("qtd_voos"),
        pl.col("ARRIVAL_DELAY").mean().alias("media_arrival_delay"),
        pl.col("ARRIVAL_DELAY").median().alias("mediana_arrival_delay"),
        pl.col("ARRIVAL_DELAY").std().alias("std_arrival_delay"),
        (pl.col("ARRIVAL_DELAY") > 0).mean().alias("taxa_atraso"),
        (pl.col("ARRIVAL_DELAY") > 15).mean().alias("taxa_atraso_grave"),
        (pl.col("ARRIVAL_DELAY") > 60).mean().alias("taxa_atraso_critico"),
        pl.col("ARRIVAL_DELAY").quantile(0.95).alias("p95_arrival_delay"),
        pl.col("DEPARTURE_DELAY").mean().alias("media_departure_delay"),
        pl.col("DISTANCE").mean().alias("distancia_media"),
        pl.col("SCHEDULED_TIME").mean().alias("tempo_voo_medio"),
        pl.col("AIRLINE").n_unique().alias("n_airlines"),
        pl.col("DESTINATION_AIRPORT").n_unique().alias("n_destinos"),
        (pl.col("periodo_dia") == "manha").mean().alias("pct_manha"),
        (pl.col("periodo_dia") == "tarde").mean().alias("pct_tarde"),
        (pl.col("periodo_dia") == "noite").mean().alias("pct_noite"),
    ])
    .filter(pl.col("qtd_voos") >= MIN_VOOS)
    .to_pandas()
    .set_index("ORIGIN_AIRPORT")
)

print(f"Aeroportos: {len(airport_features)}")
airport_features.head(3)

In [ ]:
feature_cols = [
    "media_arrival_delay", "mediana_arrival_delay", "std_arrival_delay",
    "taxa_atraso", "taxa_atraso_grave", "taxa_atraso_critico",
    "p95_arrival_delay", "media_departure_delay",
    "distancia_media", "tempo_voo_medio",
    "n_airlines", "n_destinos",
    "pct_manha", "pct_tarde", "pct_noite",
]

X = airport_features[feature_cols].fillna(airport_features[feature_cols].median())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=feature_cols)

print(f"Shape: {X_scaled.shape}")

## 2. Isolation Forest

In [ ]:
# contamination: proporção esperada de anomalias
# 0.05 = sinaliza os 5% mais isolados como anômalos
CONTAMINATION = 0.05

iso = IsolationForest(
    n_estimators=200,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iso.fit(X_scaled)

# predict: 1 = normal, -1 = anomalia
airport_features["iso_pred"] = iso.predict(X_scaled)
# score_samples: quanto mais negativo, mais anômalo
airport_features["iso_score"] = iso.score_samples(X_scaled)
airport_features["iso_anomalia"] = airport_features["iso_pred"] == -1

n_anomalos = airport_features["iso_anomalia"].sum()
print(f"Anomalias detectadas pelo Isolation Forest: {n_anomalos} de {len(airport_features)}")
print()
print("Top 10 mais anômalos:")
(
    airport_features
    .sort_values("iso_score")
    .head(10)
    [["taxa_atraso", "media_arrival_delay", "std_arrival_delay",
      "n_destinos", "qtd_voos", "iso_score"]]
)

## 3. Local Outlier Factor (LOF)

In [ ]:
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=CONTAMINATION,
    n_jobs=-1,
)

lof_pred = lof.fit_predict(X_scaled)
lof_scores = -lof.negative_outlier_factor_  # positivo = mais anômalo

airport_features["lof_pred"] = lof_pred
airport_features["lof_score"] = lof_scores
airport_features["lof_anomalia"] = lof_pred == -1

n_lof = airport_features["lof_anomalia"].sum()
print(f"Anomalias detectadas pelo LOF: {n_lof} de {len(airport_features)}")
print()
print("Top 10 mais anômalos pelo LOF:")
(
    airport_features
    .sort_values("lof_score", ascending=False)
    .head(10)
    [["taxa_atraso", "media_arrival_delay", "std_arrival_delay",
      "n_destinos", "qtd_voos", "lof_score"]]
)

## 4. Silhouette individual — aeroportos mal alocados pelo KMeans

In [ ]:
# Roda KMeans com o mesmo k do notebook 04
# Se salvou o modelo: km = joblib.load("../models/kmeans_aeroportos.pkl")
K_FINAL = 4  # ajuste conforme notebook 04

km = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init="auto")
airport_features["cluster"] = km.fit_predict(X_scaled)

sil_values = silhouette_samples(X_scaled, airport_features["cluster"])
airport_features["silhouette"] = sil_values

# Limiar: silhouette < 0 significa que o aeroporto estaria melhor em outro cluster
SILHOUETTE_THRESHOLD = 0.0
airport_features["sil_anomalia"] = airport_features["silhouette"] < SILHOUETTE_THRESHOLD

n_sil = airport_features["sil_anomalia"].sum()
print(f"Aeroportos mal alocados (silhouette < {SILHOUETTE_THRESHOLD}): {n_sil}")
print()
print("Piores silhouettes:")
(
    airport_features
    .sort_values("silhouette")
    .head(10)
    [["cluster", "silhouette", "taxa_atraso", "media_arrival_delay", "n_destinos"]]
)

## 5. Consenso — aeroportos sinalizados por múltiplos métodos

In [ ]:
airport_features["n_metodos_anomalia"] = (
    airport_features["iso_anomalia"].astype(int)
    + airport_features["lof_anomalia"].astype(int)
    + airport_features["sil_anomalia"].astype(int)
)

print("Distribuição por número de métodos que sinalizaram anomalia:")
print(airport_features["n_metodos_anomalia"].value_counts().sort_index())
print()

# Anomalias confirmadas: sinalizadas por pelo menos 2 métodos
anomalos_consenso = airport_features[airport_features["n_metodos_anomalia"] >= 2].copy()
print(f"Anomalias por consenso (≥2 métodos): {len(anomalos_consenso)}")
print()
(
    anomalos_consenso
    .sort_values("n_metodos_anomalia", ascending=False)
    [["n_metodos_anomalia", "cluster", "taxa_atraso", "taxa_atraso_grave",
      "media_arrival_delay", "std_arrival_delay", "n_destinos", "qtd_voos"]]
    .round(3)
)

## 6. Visualizações

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X_scaled)
var1, var2 = pca.explained_variance_ratio_ * 100

airport_features["PC1"] = X_2d[:, 0]
airport_features["PC2"] = X_2d[:, 1]

print(f"PC1={var1:.1f}%  PC2={var2:.1f}%  Total={var1+var2:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

METHOD_COLS = [
    ("iso_anomalia", "iso_score", "Isolation Forest", "Score IF (↓ = mais anômalo)"),
    ("lof_anomalia", "lof_score", "Local Outlier Factor", "LOF Score (↑ = mais anômalo)"),
    ("sil_anomalia", "silhouette", "KMeans Silhouette", "Silhouette (↓ = mal alocado)"),
]

for ax, (flag_col, score_col, title, cb_label) in zip(axes, METHOD_COLS):
    normal = airport_features[~airport_features[flag_col]]
    anomalo = airport_features[airport_features[flag_col]]

    sc = ax.scatter(
        normal["PC1"], normal["PC2"],
        c=normal[score_col], cmap="RdYlGn",
        s=40, alpha=0.7, edgecolors="none", label="Normal"
    )
    ax.scatter(
        anomalo["PC1"], anomalo["PC2"],
        c="red", marker="X", s=120, zorder=5,
        edgecolors="black", linewidths=0.5, label="Anomalia"
    )

    # Anotar os top 5 mais anômalos
    ascending = score_col in ["iso_score", "silhouette"]
    top5 = airport_features.sort_values(score_col, ascending=ascending).head(5)
    for ap, row in top5.iterrows():
        ax.annotate(ap, (row["PC1"], row["PC2"]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")

    plt.colorbar(sc, ax=ax, label=cb_label, shrink=0.8)
    ax.set_title(title)
    ax.set_xlabel(f"PC1 ({var1:.1f}%)")
    ax.set_ylabel(f"PC2 ({var2:.1f}%)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

plt.suptitle("Detecção de Anomalias — Projeção PCA 2D", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Mapa de consenso: 0 = normal, 1 = 1 método, 2 = 2 métodos, 3 = todos
cmap = plt.cm.get_cmap("RdYlGn_r", 4)
cores = [cmap(i / 3) for i in airport_features["n_metodos_anomalia"]]

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(
    airport_features["PC1"], airport_features["PC2"],
    c=airport_features["n_metodos_anomalia"],
    cmap="RdYlGn_r", vmin=0, vmax=3,
    s=60, alpha=0.8, edgecolors="white", linewidths=0.3
)

# Anotar aeroportos de consenso (≥2 métodos)
for ap, row in anomalos_consenso.iterrows():
    ax.annotate(
        ap, (row["PC1"], row["PC2"]),
        fontsize=8, fontweight="bold",
        xytext=(5, 5), textcoords="offset points",
        arrowprops=dict(arrowstyle="-", color="gray", lw=0.7),
    )

cbar = plt.colorbar(sc, ax=ax, ticks=[0, 1, 2, 3])
cbar.set_label("Nº de métodos que sinalizaram anomalia")
cbar.ax.set_yticklabels(["0 — Normal", "1 método", "2 métodos", "3 métodos"])

ax.set_title("Consenso de Anomalias — Projeção PCA 2D", fontsize=12)
ax.set_xlabel(f"PC1 ({var1:.1f}%)")
ax.set_ylabel(f"PC2 ({var2:.1f}%)")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Comparação dos scores dos dois principais métodos
fig, ax = plt.subplots(figsize=(10, 6))

consenso_mask = airport_features["n_metodos_anomalia"] >= 2
colors_dot = np.where(consenso_mask, "red", "steelblue")

ax.scatter(
    airport_features["iso_score"],
    airport_features["lof_score"],
    c=colors_dot, s=50, alpha=0.75, edgecolors="white", linewidths=0.3
)

# Linhas de corte dos modelos
iso_threshold = airport_features.loc[airport_features["iso_anomalia"], "iso_score"].max()
lof_threshold = airport_features.loc[airport_features["lof_anomalia"], "lof_score"].min()
ax.axvline(iso_threshold, color="orange", linestyle="--", lw=1, label=f"Corte IF ({iso_threshold:.3f})")
ax.axhline(lof_threshold, color="purple", linestyle="--", lw=1, label=f"Corte LOF ({lof_threshold:.2f})")

# Anotar consenso
for ap, row in anomalos_consenso.iterrows():
    ax.annotate(ap, (row["iso_score"], row["lof_score"]),
                fontsize=8, xytext=(5, 5), textcoords="offset points")

legend_patches = [
    mpatches.Patch(color="red", label="Anomalia consenso (≥2 métodos)"),
    mpatches.Patch(color="steelblue", label="Normal"),
]
ax.legend(handles=legend_patches + ax.get_legend_handles_labels()[0][2:])
ax.set_xlabel("Isolation Forest score (↓ = mais anômalo)")
ax.set_ylabel("LOF score (↑ = mais anômalo)")
ax.set_title("Isolation Forest vs LOF — scores por aeroporto")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Perfil dos aeroportos anômalos vs normais
cols_compare = [
    "taxa_atraso", "taxa_atraso_grave", "taxa_atraso_critico",
    "media_arrival_delay", "std_arrival_delay", "p95_arrival_delay",
    "n_destinos", "n_airlines",
]

airport_features["grupo"] = np.where(
    airport_features["n_metodos_anomalia"] >= 2, "Anômalo", "Normal"
)

perfil = (
    airport_features
    .groupby("grupo")[cols_compare]
    .mean()
    .round(3)
    .T
)

# Heatmap comparativo
perfil_norm = (perfil - perfil.min(axis=1).values.reshape(-1, 1)) / (
    perfil.max(axis=1) - perfil.min(axis=1) + 1e-9
).values.reshape(-1, 1)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    perfil_norm, annot=perfil, fmt=".3f",
    cmap="RdYlGn_r", ax=ax,
    linewidths=0.5, cbar=False
)
ax.set_title("Perfil médio: Anômalos vs Normais")
ax.set_xlabel("Grupo")
plt.tight_layout()
plt.show()

print(perfil)

## 7. Conclusões

*(Preencha após rodar com seus dados reais)*

### O que caracteriza os aeroportos anômalos?

Analise o perfil da seção 6 e responda:
- São aeroportos com **atraso muito acima** da média? (outliers de alta performance negativa)
- Ou são aeroportos com **padrão operacional diferente** — ex: muitos destinos, voos longos — independentemente do atraso?
- Existe sobreposição com aeroportos já sinalizados no EDA como problemáticos?

### Diferença entre os métodos

| Método | O que detecta | Limitação |
|--------|---------------|-----------|
| Isolation Forest | Pontos isolados no espaço de features | Não considera densidade local |
| LOF | Pontos com densidade local muito menor que seus vizinhos | Sensível ao `n_neighbors` |
| Silhouette | Aeroportos mal alocados pelo KMeans | Depende da qualidade do clustering |

### Limitações
- `contamination=0.05` é arbitrário — ajuste conforme o contexto
- Trabalhamos com dados agregados: um aeroporto pode parecer normal na média mas ter dias muito ruins
- Aeroportos com poucos voos foram filtrados — podem existir anomalias reais entre eles

### Próximos passos
- Cruzar os aeroportos anômalos com o mapa geográfico (notebook 05) — há concentração regional?
- Analisar sazonalidade dos anômalos: o comportamento atípico é constante ou concentrado em certos meses?
- Investigar os voos individuais desses aeroportos para entender a causa raiz